# Ensemble coverage and probability-of-exceedance Brier score

This notebook calculates per-case ensemble coverage and probability-of-exceedance Brier scores, then aggregates and maps them. Both metrics retain the forecast initialization `time` and `prediction_timedelta` dimensions. The two-dimensional `verification_time` coordinate records the observation timestamp used for every initialization/lead-time pair.

The synthetic input cell below makes the notebook runnable as-is. Replace it with your forecast and observation `xarray.DataArray` objects when using real data.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import cartopy.crs as ccrs

from heatextremes.metrics import coverage, probability_of_exceedance_brier_score


## Input arrays

For real data, replace this cell with `model_ensemble`, `observations`, and `threshold`. The forecast must have `time`, `prediction_timedelta`, `number`, `latitude`, and `longitude` dimensions; observations need `time`, `latitude`, and `longitude`. `threshold` may be a number or a broadcastable `DataArray` (such as the spatial threshold used here).

In [ ]:
rng = np.random.default_rng(42)

initialization_time = np.arange(
    np.datetime64("2024-07-01"), np.datetime64("2024-07-07"), np.timedelta64(1, "D")
).astype("datetime64[ns]")
prediction_timedelta = np.arange(5).astype("timedelta64[D]").astype("timedelta64[ns]")
observation_time = np.arange(
    np.datetime64("2024-07-01"), np.datetime64("2024-07-11"), np.timedelta64(1, "D")
).astype("datetime64[ns]")
latitude = np.linspace(-60, 60, 25)
longitude = np.arange(0, 360, 5)  # Deliberately 0--360: normalized below.
number = np.arange(20)

latitude_signal = 4 * np.cos(np.deg2rad(latitude))[None, None, None, :, None]
longitude_signal = 2 * np.sin(np.deg2rad(longitude))[None, None, None, None, :]
initialization_signal = np.arange(initialization_time.size)[:, None, None, None, None] / 4
lead_signal = np.arange(prediction_timedelta.size)[None, :, None, None, None] / 3
model_ensemble = xr.DataArray(
    27 + latitude_signal + longitude_signal + initialization_signal + lead_signal
    + rng.normal(scale=2.5, size=(6, 5, 20, 25, 72)),
    dims=("time", "prediction_timedelta", "number", "latitude", "longitude"),
    coords={
        "time": initialization_time,
        "prediction_timedelta": prediction_timedelta,
        "number": number,
        "latitude": latitude,
        "longitude": longitude,
    },
    name="2m_temperature",
).chunk({"time": 1, "number": 10})

observation_latitude_signal = 4 * np.cos(np.deg2rad(latitude))[None, :, None]
observation_longitude_signal = 2 * np.sin(np.deg2rad(longitude))[None, None, :]
observations = xr.DataArray(
    27 + observation_latitude_signal + observation_longitude_signal
    + np.arange(observation_time.size)[:, None, None] / 4
    + rng.normal(scale=2.0, size=(10, 25, 72)),
    dims=("time", "latitude", "longitude"),
    coords={"time": observation_time, "latitude": latitude, "longitude": longitude},
    name="2m_temperature",
)

threshold = xr.DataArray(
    30 + 2 * np.cos(np.deg2rad(latitude))[:, None] + np.zeros_like(longitude)[None, :],
    dims=("latitude", "longitude"),
    coords={"latitude": latitude, "longitude": longitude},
    name="temperature_threshold",
)

# In real use, a scalar threshold is also valid, for example: threshold = 35.0


## Normalize and align the spatial grid

The metrics use xarray alignment for non-time dimensions. This cell normalizes all longitudes to the conventional half-open `[-180, 180)` range, sorts them, and limits the arrays to their shared spatial grid.

In [ ]:
def normalize_longitude(data: xr.DataArray) -> xr.DataArray:
    """Convert a longitude coordinate from 0--360 to [-180, 180) and sort it."""
    if "longitude" not in data.coords:
        raise ValueError("All inputs must have a longitude coordinate")
    normalized_longitude = (data.longitude + 180) % 360 - 180
    if normalized_longitude.to_index().has_duplicates:
        raise ValueError("Longitude normalization produced duplicate coordinates")
    return data.assign_coords(longitude=normalized_longitude).sortby("longitude")

model_ensemble = normalize_longitude(model_ensemble)
observations = normalize_longitude(observations)
if isinstance(threshold, xr.DataArray):
    threshold = normalize_longitude(threshold)

# Do not align forecast initialization times with observation times: observations
# are matched later at initialization time + lead time by the metric functions.
excluded_dimensions = {"time", "prediction_timedelta", "number"}
model_ensemble, observations = xr.align(
    model_ensemble, observations, join="inner", exclude=excluded_dimensions, copy=False
)
if isinstance(threshold, xr.DataArray):
    model_ensemble, observations, threshold = xr.align(
        model_ensemble, observations, threshold,
        join="inner",
        exclude=excluded_dimensions,
        copy=False,
    )

xr.testing.assert_identical(model_ensemble.latitude, observations.latitude)
xr.testing.assert_identical(model_ensemble.longitude, observations.longitude)
if isinstance(threshold, xr.DataArray):
    for coordinate in ("latitude", "longitude"):
        if coordinate in threshold.coords:
            xr.testing.assert_identical(model_ensemble[coordinate], threshold[coordinate])
assert model_ensemble.longitude.min().item() >= -180
assert model_ensemble.longitude.max().item() < 180
model_ensemble


## Calculate per-case scores

Each output retains `time`, `prediction_timedelta`, `latitude`, and `longitude`; only `number` is reduced. This leaves the choice of aggregation—by lead, initialization, season, or spatial region—to downstream code.

In [ ]:
case_coverage = coverage(model_ensemble, observations, percentile=90)
case_brier_score = probability_of_exceedance_brier_score(
    model_ensemble, observations, threshold
)

required_dimensions = {"time", "prediction_timedelta", "latitude", "longitude"}
assert required_dimensions.issubset(case_coverage.dims)
assert required_dimensions.issubset(case_brier_score.dims)
assert "number" not in case_coverage.dims
assert "number" not in case_brier_score.dims

print(case_coverage)
print(case_brier_score)


## Aggregate and map

These maps average over both initialization and lead time. Change `aggregation_dimensions` to preserve one or both dimensions, or use `groupby` for a different diagnostic.

In [ ]:
aggregation_dimensions = ("time", "prediction_timedelta")
coverage_map = case_coverage.mean(aggregation_dimensions, skipna=True)
brier_score_map = case_brier_score.mean(aggregation_dimensions, skipna=True)

figure, axes = plt.subplots(
    ncols=2, figsize=(16, 5), subplot_kw={"projection": ccrs.PlateCarree()}
)

coverage_map.plot(
    ax=axes[0],
    x="longitude",
    y="latitude",
    transform=ccrs.PlateCarree(),
    cmap="viridis",
    vmin=0,
    vmax=1,
    cbar_kwargs={"label": "90% interval coverage"},
)
brier_score_map.plot(
    ax=axes[1],
    x="longitude",
    y="latitude",
    transform=ccrs.PlateCarree(),
    cmap="magma_r",
    vmin=0,
    vmax=1,
    cbar_kwargs={"label": "Brier score"},
)

for axis, title in zip(
    axes,
    ("Mean central-90% ensemble coverage", "Mean exceedance Brier score"),
):
    axis.set_global()
    axis.coastlines(linewidth=0.7)
    gridlines = axis.gridlines(draw_labels=True, linewidth=0.4, color="gray", alpha=0.5)
    gridlines.top_labels = False
    gridlines.right_labels = False
    axis.set_title(title)

figure.tight_layout()


## Example: aggregate by lead time

Because the case-wise results still have `time` and `prediction_timedelta`, aggregating over space and initialization produces lead-time diagnostics without recalculating the metrics.

In [ ]:
mean_coverage_by_lead = case_coverage.mean(("time", "latitude", "longitude"), skipna=True)
mean_brier_by_lead = case_brier_score.mean(("time", "latitude", "longitude"), skipna=True)
lead_days = mean_coverage_by_lead.prediction_timedelta / np.timedelta64(1, "D")

figure, axis = plt.subplots(figsize=(8, 4))
axis.plot(lead_days, mean_coverage_by_lead, marker="o", label="90% coverage")
axis.plot(lead_days, mean_brier_by_lead, marker="o", label="Exceedance Brier score")
axis.set(xlabel="Lead time (days)", ylabel="Mean score", ylim=(0, 1))
axis.grid()
axis.legend()
figure.tight_layout()
